In [9]:

import numpy as np
import pandas as pd
from scipy.linalg import eigvals


# ============================================================
# 1. DATA
# ============================================================

path = "/lustre/geocean/WORK/users/alonsoap/projects/TWINPEACS/02_Emulator_Testing/two_stage_model/outputs/pca/pcs_msl_sel_6h.csv"

df = pd.read_csv(path, index_col=0, parse_dates=True)

'''
pc_cols = [f"msl_PC{i}" for i in range(1, 31)]
X = df[pc_cols].dropna()
'''
pc_cols = df.columns.to_list()
X = df.dropna()
print("Observations:", len(X))
print("Variables:", len(pc_cols))
print("Period:", X.index.min(), "-", X.index.max())


# ============================================================
# 2. MULTIVARIATE SETAR
# ============================================================

def fit_setar(
    X,
    weights=None,
    p=1,
    d=1,
    min_regime_frac=0.15,
    n_thresholds=41,
):
    """
    Two-regime multivariate SETAR.

    X       : DataFrame (time x variables)
    weights : switching-index weights
    p       : VAR order
    d       : switching delay (6-hourly observations)
    """

    X = np.asarray(X, dtype=float)

    n, m = X.shape

    if weights is None:
        weights = np.eye(m)[0]

    weights = np.asarray(weights, dtype=float)
    weights = weights / np.linalg.norm(weights)

    # Standardise ONLY the switching index
    x_mean = X.mean(axis=0)
    x_std = X.std(axis=0, ddof=0)
    x_std[x_std == 0] = 1.0

    X_scaled = (X - x_mean) / x_std

    # Switching index
    Z = X_scaled @ weights

    # Construct regression matrices
    start = max(p, d)

    Y = X[start:]

    lags = [
        X[start-i:n-i]
        for i in range(1, p+1)
    ]

    R = np.column_stack([
        np.ones(len(Y)),
        *lags
    ])

    Z = Z[start:]

    nobs = len(Y)
    k = R.shape[1]

    # Sort observations by switching index
    order = np.argsort(Z)

    R = R[order]
    Y = Y[order]
    Z_sorted = Z[order]

    YTY = Y.T @ Y

    # Candidate threshold positions
    lo = int(min_regime_frac * nobs)
    hi = int((1-min_regime_frac) * nobs)

    splits = np.unique(
        np.linspace(lo, hi, n_thresholds).astype(int)
    )

    best = None

    for split in splits:

        R1, R2 = R[:split], R[split:]
        Y1, Y2 = Y[:split], Y[split:]

        # Separate VAR estimation
        B1 = np.linalg.lstsq(R1, Y1, rcond=None)[0]
        B2 = np.linalg.lstsq(R2, Y2, rcond=None)[0]

        # Residual sums of squares
        E1 = Y1 - R1 @ B1
        E2 = Y2 - R2 @ B2

        # Common residual covariance
        Sigma = (
            E1.T @ E1 + E2.T @ E2
        ) / nobs

        Sigma = (Sigma + Sigma.T) / 2

        # Gaussian likelihood objective
        sign, logdet = np.linalg.slogdet(Sigma)

        if sign <= 0:
            continue

        # Number of estimated parameters
        n_params = (
            2 * k * m
            + m * (m+1) // 2
            + 1
        )

        bic = nobs * logdet + n_params * np.log(nobs)
        aic = nobs * logdet + 2 * n_params

        if best is None or bic < best["BIC"]:

            threshold = (
                Z_sorted[split-1] + Z_sorted[split]
            ) / 2

            best = {
                "B1": B1,
                "B2": B2,
                "threshold": threshold,
                "weights": weights,
                "delay": d,
                "order": p,
                "Sigma": Sigma,
                "BIC": bic,
                "AIC": aic,
                "regime_fraction": (
                    split/nobs,
                    1-split/nobs
                ),
                "nobs": nobs,
                "x_mean": x_mean,
                "x_std": x_std,
            }

    if best is None:
        raise RuntimeError("No valid threshold found.")

    return best


# ============================================================
# 3. DIAGNOSTICS
# ============================================================

def print_results(model):

    print("\n" + "="*55)
    print("MULTIVARIATE SETAR RESULTS")
    print("="*55)

    print("VAR order:", model["order"])
    print("Switching delay:", model["delay"], "steps")

    print("Threshold:", model["threshold"])

    print(
        "Regime 1 occupancy:",
        f"{100*model['regime_fraction'][0]:.1f}%"
    )

    print(
        "Regime 2 occupancy:",
        f"{100*model['regime_fraction'][1]:.1f}%"
    )

    print("AIC:", model["AIC"])
    print("BIC:", model["BIC"])

    m = model["Sigma"].shape[0]

    for k in (1, 2):

        B = model[f"B{k}"]

        # Companion matrix for VAR stability
        A = B[1:].T.reshape(
            model["order"], m, m
        )

        companion = np.zeros(
            (m*model["order"], m*model["order"])
        )

        companion[:m, :] = np.hstack(A)

        if model["order"] > 1:
            companion[m:, :-m] = np.eye(
                m*(model["order"]-1)
            )

        rho = np.max(np.abs(eigvals(companion)))

        print(f"Regime {k} spectral radius: {rho:.4f}")


# ============================================================
# 4. SIMULATION
# ============================================================

def simulate_setar(model, n_steps, initial_values=None, seed=1234):

    rng = np.random.default_rng(seed)

    p = model["order"]
    m = model["Sigma"].shape[0]

    if initial_values is None:
        history = [
            row.copy() for row in
            np.zeros((p, m))
        ]
    else:
        history = [
            row.copy() for row in
            np.asarray(initial_values)[-p:]
        ]

    output = np.zeros((n_steps, m))
    regimes = np.zeros(n_steps, dtype=int)

    for t in range(n_steps):

        # Switching index from past atmospheric state
        x_switch = history[-model["delay"]]

        z = (
            (x_switch - model["x_mean"])
            / model["x_std"]
        ) @ model["weights"]

        if z <= model["threshold"]:
            B = model["B1"]
            regimes[t] = 1
        else:
            B = model["B2"]
            regimes[t] = 2

        # Construct lag vector
        predictors = np.concatenate([
            [1.0],
            *[history[-i] for i in range(1, p+1)]
        ])

        # Generate next PC vector
        noise = rng.multivariate_normal(
            np.zeros(m),
            model["Sigma"]
        )

        x_new = predictors @ B + noise

        output[t] = x_new

        history.append(x_new)

    return output, regimes


Observations: 126864
Variables: 30
Period: 1940-01-01 00:00:00 - 2026-10-31 18:00:00


In [10]:

# Candidate switching indices
m = len(pc_cols)

weights_to_test = {
    "PC1": np.eye(m)[0],

    "PC1_PC2": np.array(
        [1, 1] + [0]*(m-2), dtype=float
    ),

    "PC1_PC3": np.array(
        [1, 1, 1] + [0]*(m-3), dtype=float
    ),

    "PC1_PC5": np.array(
        [1]*5 + [0]*(m-5), dtype=float
    ),
}

results = {}

for name, weights in weights_to_test.items():

    for p in [1, 2, 3]:

        print(f"Fitting {name}, VAR({p})...")

        model = fit_setar(
            X,
            weights=weights,
            p=p,
            d=1,
            min_regime_frac=0.15,
            n_thresholds=41,
        )

        results[(name, p)] = model

        print(
            f"BIC: {model['BIC']:.2f} | "
            f"Threshold: {model['threshold']:.3f}"
        )


# Compare models
comparison = pd.DataFrame([
    {
        "Switching index": name,
        "VAR order": p,
        "BIC": model["BIC"],
        "AIC": model["AIC"],
        "Regime 1 (%)": 100*model["regime_fraction"][0],
        "Regime 2 (%)": 100*model["regime_fraction"][1],
    }
    for (name, p), model in results.items()
]).sort_values("BIC")

print(comparison.to_string(index=False))


Fitting PC1, VAR(1)...
BIC: -696765.53 | Threshold: -0.375
Fitting PC1, VAR(2)...
BIC: -2506473.80 | Threshold: -0.318
Fitting PC1, VAR(3)...
BIC: -2848441.91 | Threshold: -0.375
Fitting PC1_PC2, VAR(1)...
BIC: -707390.67 | Threshold: -0.170
Fitting PC1_PC2, VAR(2)...
BIC: -2515115.65 | Threshold: -0.129
Fitting PC1_PC2, VAR(3)...
BIC: -2856814.83 | Threshold: -0.170
Fitting PC1_PC3, VAR(1)...
BIC: -706953.58 | Threshold: -0.360
Fitting PC1_PC3, VAR(2)...
BIC: -2512029.66 | Threshold: -0.237
Fitting PC1_PC3, VAR(3)...
BIC: -2853225.69 | Threshold: -0.405
Fitting PC1_PC5, VAR(1)...
BIC: -715011.94 | Threshold: -0.574
Fitting PC1_PC5, VAR(2)...
BIC: -2510060.33 | Threshold: -0.574
Fitting PC1_PC5, VAR(3)...
BIC: -2849548.98 | Threshold: -0.574
Switching index  VAR order           BIC           AIC  Regime 1 (%)  Regime 2 (%)
        PC1_PC2          3 -2.856815e+06 -2.914598e+06     42.999030     57.000970
        PC1_PC3          3 -2.853226e+06 -2.911009e+06     32.499350     67.500650

In [11]:

# Select model
model = results[("PC1", 1)]

print_results(model)

# Important: initialise with enough historical observations
# for both the VAR lags and switching delay.
initial = X.iloc[-max(model["order"], model["delay"]):].values

# Simulate 100 years at 6-hourly resolution
n_years = 100
n_steps = n_years * 365 * 4

# In simulate_setar, initialise history using max(p, d)
# rather than just p observations.

synthetic, regimes = simulate_setar(
    model,
    n_steps=n_steps,
    initial_values=initial,
    seed=1234
)

synthetic_df = pd.DataFrame(
    synthetic,
    columns=pc_cols
)

synthetic_df.to_csv("setar_synthetic.csv")

print(synthetic_df.describe())
print(pd.Series(regimes).value_counts(normalize=True))



MULTIVARIATE SETAR RESULTS
VAR order: 1
Switching delay: 1 steps
Threshold: -0.37490556104122164
Regime 1 occupancy: 30.7%
Regime 2 occupancy: 69.3%
AIC: -719446.0343970029
BIC: -696765.5269589283
Regime 1 spectral radius: 0.9739
Regime 2 spectral radius: 0.9853
             msl_PC4        msl_PC1        msl_PC3       msl_PC11  \
count  146000.000000  146000.000000  146000.000000  146000.000000   
mean       -1.495903       0.507257      -1.470621       0.083399   
std        11.604980      24.827102      16.854447       3.841151   
min       -44.296324    -112.098259     -74.588268     -15.531784   
25%       -10.653842     -16.481206     -12.807740      -2.512277   
50%        -0.402430       0.235077      -1.532053       0.047240   
75%         7.250910      17.517817       9.916980       2.649671   
max        37.951888      94.564404      67.075194      17.239980   

             msl_PC6        msl_PC2       msl_PC34        msl_PC8  \
count  146000.000000  146000.000000  146000.0

In [16]:
# Assuming model contains the fitted threshold and switching index
# and X is your original PC DataFrame.

X_scaled = (X - model["x_mean"]) / model["x_std"]
Z = X_scaled @ model["weights"]

regime = (Z.shift(model["delay"]) > model["threshold"]).astype(int) + 1

# Mean atmospheric state in each regime
print(X.groupby(regime).mean().T)

# Standard deviation in each regime
print(X.groupby(regime).std().T)

# Seasonal distribution
season = X.index.month.map({
    12: "DJF", 1: "DJF", 2: "DJF",
    3: "MAM", 4: "MAM", 5: "MAM",
    6: "JJA", 7: "JJA", 8: "JJA",
    9: "SON", 10: "SON", 11: "SON"
})

print(pd.crosstab(regime, season, normalize="index"))

                  1         2
msl_PC4  -13.256481  5.597192
msl_PC1   -0.228249 -1.726033
msl_PC3   -0.027833 -0.062585
msl_PC11   0.084948 -0.056137
msl_PC6   -1.781734  0.244424
msl_PC2   -0.973393  0.586677
msl_PC34   0.011399  0.015579
msl_PC8   -0.488582  0.340317
msl_PC27   0.033613  0.038656
msl_PC38  -0.005058  0.010858
msl_PC17   0.020103 -0.086331
msl_PC30  -0.052996 -0.052289
msl_PC13   0.222286 -0.036263
msl_PC42   0.010122  0.004242
msl_PC36   0.033517 -0.004116
msl_PC45  -0.029243 -0.024342
msl_PC12  -0.032857 -0.077390
msl_PC23   0.109881 -0.072846
msl_PC43  -0.012668  0.017321
msl_PC41  -0.004727 -0.030979
msl_PC32   0.036973  0.037213
msl_PC18   0.059175 -0.007966
msl_PC35   0.013734 -0.026696
msl_PC50  -0.016339 -0.004188
msl_PC7    0.000957 -0.119143
msl_PC20   0.005850 -0.027054
msl_PC21  -0.082205  0.040379
msl_PC5    0.044130  0.379798
msl_PC24   0.121785  0.025467
msl_PC9   -0.115410 -0.124677
                  1          2
msl_PC4    8.028796   6.667371
msl_PC1 

In [15]:
model.keys()

dict_keys(['B1', 'B2', 'threshold', 'weights', 'delay', 'order', 'Sigma', 'BIC', 'AIC', 'regime_fraction', 'nobs', 'x_mean', 'x_std'])